# 第十章 · 综合成长补充因子

## 研报核心观点

将前几章的因子**融合成一个综合因子**，避免单一因子的偏差：
- 成长路径综合
- 成长动能综合
- 未来成长
- 超预期成长
- 成长性价比
- 多维成长能力

## 公式

**综合成长补充因子 = 等权融合上述 6 类因子**

各因子已标准化（去极值 + z-score），等权平均。

## 预期效果

| 因子 | Rank IC | ICIR | 多空年化 |
|------|---------|------|---------|
| **综合成长补充因子** | **6.03%** | **3.24** | **24.95%** |
| 残差因子（vs 传统成长）| 5.41% | 3.09 | 15.61% |

残差因子 = 综合成长补充因子对传统成长因子做回归的**残差**，
衡量新增因子相对传统成长是否有**增量信息**。

## 1. 环境设置

In [ ]:
import sys
sys.path.insert(0, '.')
import 因子库

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪')

## 2. 加载所有子因子

为了减少重复计算，我们在这里直接**调用前几章定义好的因子**。

由于 notebook 是独立的，我们重新加载基础数据并计算：

In [ ]:
# 加载基础数据
pit_profit = 因子库.load_pit_financial('net_profit_parent_company')
pit_revenue = 因子库.load_pit_financial('operating_revenue')
pit_assets = 因子库.load_pit_financial('total_assets')
pit_gross = 因子库.load_pit_financial('gross_profit')
pit_cfo = 因子库.load_pit_financial('cash_flow_from_operating_activities')
pit_equity = 因子库.load_pit_financial('equity_parent_company')

sq_profit, _ = 因子库.to_single_quarter(pit_profit, 'net_profit_parent_company')
sq_revenue, _ = 因子库.to_single_quarter(pit_revenue, 'operating_revenue')
sq_assets, _ = 因子库.to_single_quarter(pit_assets, 'total_assets')
sq_gross, _ = 因子库.to_single_quarter(pit_gross, 'gross_profit')
sq_cfo, _ = 因子库.to_single_quarter(pit_cfo, 'cash_flow_from_operating_activities')
sq_equity, _ = 因子库.to_single_quarter(pit_equity, 'equity_parent_company')

close_price = 因子库.load_daily('close')
returns_next_month = 因子库.next_month_return(close_price, months=1)

# 估值因子
pb_lyr = 因子库.load_daily_path(因子库.DATA_DIR / 'factor' / 'pb_ratio_lyr.pkl')
pe_lyr = 因子库.load_daily_path(因子库.DATA_DIR / 'factor' / 'pe_ratio_lyr.pkl')
pcf_lyr = 因子库.load_daily_path(因子库.DATA_DIR / 'factor' / 'pcf_ratio_lyr.pkl')

print('数据准备完成')

## 3. 因子 1：成长路径综合

第三章：位移路程比 + 上涨次数 + 回归拟合度

In [ ]:
def calc_growth_path_composite(sq_profit, sq_revenue, sq_assets):
    # 位移路程比
    def disp_ratio(sq):
        if sq_derivative.sub is None:
            return None
        n = 5
        if sq.shape[1] < n:
            return None
        sub = sq.iloc[:, -n:]
        end = sub.iloc[:, -1]
        start = sub.iloc[:, 0]
        disp = (end - start).abs()
        diffs = sub.diff(axis=1)
        path = diffs.iloc[:, 1:].abs().sum(axis=1)
        return disp / path.replace(0, np.nan)

    # 上涨次数（20期窗口）
    from scipy.stats import percentileofscore
    def up_freq(sq, window=20):
        if sq.shape[1] < window:
            return None
        diffs = sq.diff(axis=1)
        sub = diffs.iloc[:, -window:]
        up = (sub > 0).sum(axis=1)
        down = (sub < 0).sum(axis=1)
        return up / (up + down).replace(0, np.nan)

    # 回归拟合度
    def reg_r2(sq, window=5):
        if sq.shape[1] < window:
            return None
        sub = sq.iloc[:, -window:]
        x = np.arange(window)
        r2_list = []
        for idx in sub.index:
            y = sub.loc[idx].values
            if np.isnan(y).any() or np.std(y) == 0:
                r2_list.append(np.nan)
                continue
            slope, intercept, r, p, se = stats.linregress(x, y)
            r2_list.append(r ** 2)
        r2 = pd.Series(r2_list, index=sub.index)
        sign = np.sign(sub.iloc[:, -1] - sub.iloc[:, 0]).astype(float)
        sign[sign == 0] = np.nan
        return r2 * sign

    # 滚动计算因子
    dr_list, uf_list, r2_list = [], [], []
    for i in range(8, sq_profit.shape[1]):
        window_p = sq_profit.iloc[:, :i+1].dropna(axis=1, how='all')
        if window_p.shape[1] < 5:
            continue
        # 用月末日期作为因子日期
        end_date = window_p.columns[-1]
        # 计算（简化为单只窗口的最后一个值）
        sub = window_p.iloc[:, -5:]
        end = sub.iloc[:, -1]
        start = sub.iloc[:, 0]
        disp = (end - start).abs()
        diffs = sub.diff(axis=1)
        path = diffs.iloc[:, 1:].abs().sum(axis=1)
        dr = disp / path.replace(0, np.nan)
        dr.name = end_date
        dr_list.append(dr)

    for i in range(21, sq_profit.shape[1]):
        window_p = sq_profit.iloc[:, :i+1].dropna(axis=1, how='all')
        if window_p.shape[1] < 21:
            continue
        end_date = window_p.columns[-1]
        diffs = window_p.diff(axis=1)
        sub = diffs.iloc[:, -20:]
        up = (sub > 0).sum(axis=1)
        down = (sub < 0).sum(axis=1)
        uf = up / (up + down).replace(0, np.nan)
        uf.name = end_date
        uf_list.append(uf)

    for i in range(8, sq_profit.shape[1]):
        window_p = sq_profit.iloc[:, :i+1].dropna(axis=1, how='all')
        if window_p.shape[1] < 5:
            continue
        end_date = window_p.columns[-1]
        sub = window_p.iloc[:, -5:]
        x = np.arange(5)
        r2_list_inner = []
        for idx in sub.index:
            y = sub.loc[idx].values
            if np.isnan(y).any() or np.std(y) == 0:
                r2_list_inner.append(np.nan)
                continue
            slope, intercept, r, p, se = stats.linregress(x, y)
            r2_list_inner.append(r ** 2)
        r2 = pd.Series(r2_list_inner, index=sub.index)
        sign = np.sign(sub.iloc[:, -1] - sub.iloc[:, 0]).astype(float)
        sign[sign == 0] = np.nan
        r2_signed = r2 * sign
        r2_signed.name = end_date
        r2_list.append(r2_signed)

    f_dr = pd.concat(dr_list, axis=1).T
    f_uf = pd.concat(uf_list, axis=1).T
    f_r2 = pd.concat(r2_list, axis=1).T
    f_dr.index = pd.to_datetime(f_dr.index)
    f_uf.index = pd.to_datetime(f_uf.index)
    f_r2.index = pd.to_datetime(f_r2.index)

    # 标准化 + 等权
    common = f_dr.index.intersection(f_uf.index).intersection(f_r2.index)
    common_stocks = f_dr.columns
    return (
        因子库.normalize_factor(f_dr.loc[common, common_stocks]) +
        因子库.normalize_factor(f_uf.loc[common, common_stocks]) +
        因子库.normalize_factor(f_r2.loc[common, common_stocks])
    ) / 3


factor_path = calc_growth_path_composite(sq_profit, sq_revenue, sq_assets)
print(f'成长路径综合因子: {factor_path.shape}')

## 4. 因子 2：成长动能综合

第四章：加速度 + 增速分位 + 回归斜率

In [ ]:
# 净利润同比
yoy_profit = sq_profit / sq_profit.shift(4, axis=1) - 1

# 加速度（过去 4 期差值均值）
acc_profit = yoy_profit.diff(axis=1).iloc[:, -4:].mean(axis=1)
# 增速分位（过去 8 期的 percentile rank）
from scipy.stats import percentileofscore

# 简化：用最近 8 期的同比作为分位
recent_yoy = yoy_profit.iloc[:, -8:]

pct_list = []
for idx in recent_yoy.index:
    h = recent_yoy.loc[idx].dropna().values
    if len(h) < 2:
        pct_list.append(np.nan)
        continue
    pct_list.append(percentileofscore(h, h[-1]) / 100)
recent_pct = pd.Series(pct_list, index=recent_yoy.index, name=recent_yoy.columns[-1])

# 回归斜率
window = yoy_profit.iloc[:, -4:]
x = np.arange(4)
slope_list = []
for idx in window.index:
    y = window.loc[idx].values
    if np.isnan(y).any() or np.std(y) == 0:
        slope_list.append(np.nan)
        continue
    s, _, _, _, _ = stats.linregress(x, y)
    slope_list.append(s)
slope = pd.Series(slope_list, index=window.index, name=window.columns[-1])

# 标准化 + 等权
acc_norm = 因子库.normalize_factor(acc_profit.to_frame().T).T.iloc[0]
pct_norm = 因子库.normalize_factor(recent_pct.to_frame().T).T.iloc[0]
slp_norm = 因子库.normalize_factor(slope.to_frame().T).T.iloc[0]

common_idx = acc_norm.index.intersection(pct_norm.index).intersection(slp_norm.index)
common_stocks = acc_norm.index

factor_momentum = (
    acc_norm.loc[common_idx] +
    pct_norm.loc[common_idx] +
    slp_norm.loc[common_idx]
) / 3

# 转为 date x stock DataFrame
# （实际实现这里需要按月聚合，简化版用最后一个日期）
print(f'成长动能综合因子: {factor_momentum.shape}')

## 5. 因子 3：未来成长

In [ ]:
# 简化版：用过去 8 季的净利润同比作为预测代理
# （完整版需要 Ridge 回归，参考 05_future_growth.ipynb）

future_growth_proxy = yoy_profit.iloc[:, -4:].mean(axis=1)
print(f'未来成长代理: {future_growth_proxy.shape}')

## 6. 因子 4：超预期成长（SUE）

In [ ]:
# SUE（已实现）
yoy = sq_profit / sq_profit.shift(4, axis=1) - 1
expected_yoy = yoy.T.rolling(window=8).mean()
expected_profit_now = sq_profit.shift(4, axis=1) * (1 + expected_yoy)
surprise = sq_profit - expected_profit_now
volatility = yoy.T.rolling(window=8).std()
sue_factor = surprise / volatility.replace(0, np.nan)

print(f'SUE 因子: {sue_factor.shape}')

## 7. 因子 5：成长性价比

In [ ]:
# 综合估值 + 净利润增速排名
valuation_rank = (
    pb_lyr.rank(axis=1, ascending=True, pct=True) +
    pe_lyr.rank(axis=1, ascending=True, pct=True) +
    pcf_lyr.rank(axis=1, ascending=True, pct=True)
) / 3

# TTM 净利润同比（行方向 → 转置后做 rolling）
ttm_profit = sq_profit.T.rolling(window=4, min_periods=4).sum().T
ttm_yoy = ttm_profit / ttm_profit.shift(4, axis=1) - 1
ttm_yoy = ttm_yoy.dropna(axis=1, how='all')
growth_rank = ttm_yoy.rank(axis=1, pct=True)

# 成长性价比（取公共部分）
common_idx = growth_rank.index.intersection(valuation_rank.index)
common_cols = growth_rank.columns.intersection(valuation_rank.columns)
factor_gv = (
    growth_rank.loc[common_idx, common_cols] +
    valuation_rank.loc[common_idx, common_cols]
) / 2
print(f'成长性价比: {factor_gv.shape}')

## 8. 因子 6：多维成长

第八章：5 维度等权

In [ ]:
# 多维成长代表因子
dim1 = 因子库.normalize_factor(yoy_profit)
dim2 = 因子库.normalize_factor(yoy)
dim3 = 因子库.normalize_factor(sq_cfo / sq_cfo.shift(4, axis=1) - 1)
dim4 = 因子库.normalize_factor((sq_revenue / sq_assets) / (sq_revenue / sq_assets).shift(4, axis=1) - 1)
roe = sq_profit / sq_equity.replace(0, np.nan)
dim5 = 因子库.normalize_factor(roe - roe.shift(4, axis=1))

# 对齐日期
common = dim1.index
common_stocks = dim1.columns

factor_multidim = (
    dim1.loc[common, common_stocks] +
    dim2.loc[common, common_stocks] +
    dim3.loc[common, common_stocks] +
    dim4.loc[common, common_stocks] +
    dim5.loc[common, common_stocks]
) / 5
print(f'多维成长综合: {factor_multidim.shape}')

## 9. 综合成长补充因子

把所有 6 类因子的**截面值**等权融合。

（注：实际研报中因子都是月度截面，因子值已 z-score 时对）

In [ ]:
# 完整版需要月度截面数据。这里展示逻辑：
# 由于各因子矩阵shape不同（季度vs月度），完整实现需要统一为月度截面。

# 简化：用各因子的最后一个截面拼接演示
all_factors = {}

try:
    if hasattr(factor_path, 'iloc') and len(factor_path) > 0:
        all_factors['成长路径'] = factor_path.iloc[-1]
except:
    pass

try:
    if hasattr(sue_factor, 'iloc') and len(sue_factor) > 0:
        all_factors['超预期'] = sue_factor.iloc[-1]
except:
    pass

try:
    if hasattr(factor_gv, 'iloc') and len(factor_gv) > 0:
        all_factors['成长性价比'] = factor_gv.iloc[-1]
except:
    pass

print('各因子最新截面：')
for name, f in all_factors.items():
    print()
    print(name, ':')
    if hasattr(f, 'head'):
        print(f.head())

## 10. 因子汇总

| 因子 | 我们的 IC | 研报 IC | 我们的 ICIR | 研报 ICIR |
|------|----------|---------|------------|----------|
| 综合成长补充因子 | 待测 | **6.03%** | - | **3.24** |
| 残差因子 | - | 5.41% | - | 3.09 |

### 实施说明

完整实现需要：
1. **统一月度截面数据**：把所有因子转成月频 date × stock
2. **行业内中性化**：每个截面上按中信一级行业内排名
3. **等权融合** → 综合成长补充因子
4. **回归取残差**：综合 vs 传统成长的残差

### 下一步

- **第十一章**：策略回测（30 只股票组合 + 月度调仓 + 基准对比）